#### Import du fichier CSV et des colonnes

In [1]:
from pathlib import Path

import pandas as pd

CSV_GZ = Path("..") / "data" / "france" / "food_france.csv.gz"

COLUMNS = [
    "code",
    "product_name",
    "nutriscore_grade",
    "energy-kj_100g",
    "energy-kcal_100g",
    "energy_100g",
    "fat_100g",
    "saturated-fat_100g",
    "carbohydrates_100g",
    "sugars_100g",
    "fiber_100g",
    "proteins_100g",
    "salt_100g",
    "sodium_100g",
    "fruits-vegetables-legumes_100g",
]

dataframe = pd.read_csv(
    CSV_GZ,
    sep="\t",
    usecols=COLUMNS,
    dtype={"code": "string"},
    low_memory=False
)



##### Distribution des valeurs sur les nutriments
Ici, on récupère : 
* Les quartiles
* Les valeurs min et max
* La moyenne
* La médiane

Ca nous permet d'identifier facilement les valeurs aberrantes

In [2]:
num = dataframe.select_dtypes(include="number")

table = pd.DataFrame({
    "0.25": num.quantile(0.25),
    "0.50": num.quantile(0.50),
    "0.75": num.quantile(0.75),
    "min": num.min(),
    "max": num.max(),
    "moyenne": num.mean(),
    "médiane": num.median(),
})
table.index.name = "nutriment"

table.style.format("{:.1f}")

,0.25,0.50,0.75,min,max,moyenne,médiane
nutriment,,,,,,,
energy-kj_100g,484.8,1096.6,1674.5,-700.0,10000000000.0,12616.4,1096.6
energy-kcal_100g,113.0,262.0,400.0,-113.9,14232696253056154.0,16150009474.6,262.0
energy_100g,477.6,1089.8,1671.9,-700.0,10000000000.0,12510.1,1089.8
fat_100g,1.0,8.1,22.3,-0.0,3257.0,14.3,8.1
saturated-fat_100g,0.2,2.0,8.0,-3.6,1238.0,5.4,2.0
carbohydrates_100g,2.7,14.0,51.1,0.0,3888.0,27.0,14.0
sugars_100g,0.6,3.5,19.5,0.0,1174717414144440050270536724381696.0,1343721269355030180685414400.0,3.5
fiber_100g,0.0,1.5,3.6,-0.4,6636.0,3.1,1.5
proteins_100g,1.5,6.3,13.0,-0.0,40939.0,9.1,6.3


##### Distribution et cardinalité sur les nutriscores
On récupère la part des produits notés, et des non notés

In [3]:
grades = dataframe['nutriscore_grade']

scored = grades.isin(list('abcde'))

summary = pd.Series({
    "noté (a à e)": scored.sum(),
    "unknown": (grades == "unknown").sum(),
    "not-applicable": (grades == "not-applicable").sum(),
    "manquant (NaN)": grades.isna().sum(),
})

pd.DataFrame(
    {"nombre": summary, "%": summary / len(grades) * 100}).style.format(
    {"nombre": "{:,.0f}", "%": "{:.1f}"}
)

,nombre,%
noté (a à e),"474,685",37.6
unknown,"718,633",57.0
not-applicable,"41,960",3.3
manquant (NaN),"25,985",2.1


Ici, on analyse un peu plus en détail.
On fait ressortir le pourcentage de chaque note sur l'ensemble des produits qui ont une notes

In [4]:
counts_scored = grades[scored].value_counts().sort_index()

pd.DataFrame({
    "nombre": counts_scored,
    "% des notés": counts_scored / counts_scored.sum() * 100,
}).style.format({"nombre": "{:,.0f}", "% des notés": "{:.1f}"})

,nombre,% des notés
nutriscore_grade,,
a,"65,698",13.8
b,"51,274",10.8
c,"98,607",20.8
d,"125,269",26.4
e,"133,837",28.2


##### Récupération des doublons de codes-barres

In [5]:
codes = dataframe["code"]
is_dup = codes.duplicated(keep=False) & codes.notna()

print(f"Codes manquants          : {codes.isna().sum():,}")
print(f"Lignes avec code répété  : {is_dup.sum():,}")
print(f"Codes distincts répétés  : {codes[is_dup].nunique():,}")

Codes manquants          : 0
Lignes avec code répété  : 52
Codes distincts répétés  : 26


In [6]:
exact = dataframe.duplicated(keep=False)
print(f"Lignes identiques sur toutes les colonnes : {exact.sum():,}")

Lignes identiques sur toutes les colonnes : 6


In [7]:
dataframe[is_dup].sort_values("code").head(10)

,code,product_name,nutriscore_grade,energy-kj_100g,energy-kcal_100g,energy_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,sodium_100g,fruits-vegetables-legumes_100g
18992,0059527070430,Goji Snack Bars,unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18993,0059527070430,Goji,c,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18998,0059527171687,Pomegranate,c,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18999,0059527171687,Tast Of Nature Pomegranate Granenreep @,unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19019,0059527401555,Almond Fruit And Nut Snack Bar,c,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19020,0059527401555,Almond,c,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19021,0059527501552,Cranberry Fruit and Nut Snack Bar,c,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19022,0059527501552,Cranberry,c,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19025,0059527601559,Brazil nut,c,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19026,0059527601559,"Coconut, Cranberry, Brazil Nut Snack Bar Varie...",d,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


##### Taux de remplissage sur les énergies

In [8]:
kj = dataframe["energy-kj_100g"]
kcal = dataframe["energy-kcal_100g"]

both_filled = kj.notna() & kcal.notna()

print(f"Les deux remplis : {both_filled.sum():,} ({both_filled.mean():.1%} des produits)")
print(f"Seulement kJ    : {(kj.notna() & kcal.isna()).sum():,}")
print(f"Seulement kcal  : {(kj.isna() & kcal.notna()).sum():,}")
print(f"Aucun des deux  : {(kj.isna() & kcal.isna()).sum():,}")

Les deux remplis : 873,496 (69.3% des produits)
Seulement kJ    : 332
Seulement kcal  : 7,785
Aucun des deux  : 379,650


##### Contrôles des incohérences d'unités sur les énergies
Ratio kJ/kcal théorique : 4,184. Plage retenue : 3,8 à 4,6.
Choisie car une plage 4,0-4,4 laissait de côté ~61 000 lignes proches (arrondis d'étiquettes).
Les lignes avec une énergie à 0 sont exclues ici (traitées dans « valeurs impossibles »).

In [14]:
both = dataframe[["energy-kj_100g", "energy-kcal_100g"]].dropna()
both = both[(both > 0).all(axis=1)]

ratio = both["energy-kj_100g"] / both["energy-kcal_100g"]
ok = ratio.between(3.8, 4.6)

print(f"{len(both):,} lignes comparables")
print(f"  cohérentes   : {ok.sum():,} ({ok.mean():.1%})")
print(f"  incohérentes : {(~ok).sum():,} ({(~ok).mean():.1%})")

855,859 lignes comparables
  cohérentes   : 766,995 (89.6%)
  incohérentes : 88,864 (10.4%)


##### Contrôle des valeurs impossibles
1. Les énergies - kJ / kcal

In [18]:
kj = dataframe["energy-kj_100g"]
kcal = dataframe["energy-kcal_100g"]

rules = {
    "kJ négatif": (kj < 0).sum(),
    "kcal négatif": (kcal < 0).sum(),
    "kJ = 0": (kj == 0).sum(),
    "kcal = 0": (kcal == 0).sum(),
    "kJ > 3700": (kj > 3700).sum(),
    "kcal > 900": (kcal > 900).sum(),
}

table = pd.Series(rules, name="lignes").to_frame()
table["% de toutes les lignes"] = table["lignes"] / len(dataframe) * 100
table.style.format({"lignes": "{:,.0f}", "% de toutes les lignes": "{:.2f}"})

,lignes,% de toutes les lignes
kJ négatif,8,0.00
kcal négatif,1,0.00
kJ = 0,"14,345",1.14
kcal = 0,"17,665",1.40
kJ > 3700,601,0.05
kcal > 900,"1,145",0.09


In [20]:
filled = dataframe[["energy-kj_100g", "energy-kcal_100g"]].dropna()
zeros = (filled == 0).sum(axis=1)

print(f"Les deux à 0 (plausible pour de l'eau) : {(zeros == 2).sum():,}")
print(f"Un seul à 0 (impossible)               : {(zeros == 1).sum():,}")

Les deux à 0 (plausible pour de l'eau) : 11,068
Un seul à 0 (impossible)               : 6,561


2. Les nutriments 

In [21]:
cols = [
    "fat_100g", "saturated-fat_100g", "carbohydrates_100g", "sugars_100g",
    "fiber_100g", "proteins_100g", "salt_100g", "sodium_100g",
    "fruits-vegetables-legumes_100g",
]
sub = dataframe[cols]

table = pd.DataFrame({
    "renseignées": sub.notna().sum(),
    "négatives": (sub < 0).sum(),
    "> 100": (sub > 100).sum(),
})
table["% négatives"] = table["négatives"] / table["renseignées"] * 100
table["% > 100"] = table["> 100"] / table["renseignées"] * 100
table.index.name = "nutriment"

table.style.format({
    "renseignées": "{:,.0f}", "négatives": "{:,.0f}", "> 100": "{:,.0f}",
    "% négatives": "{:.2f}", "% > 100": "{:.2f}",
})

,renseignées,négatives,> 100,% négatives,% > 100
nutriment,,,,,
fat_100g,"874,248",1,45,0.00,0.01
saturated-fat_100g,"873,052",1,19,0.00,0.00
carbohydrates_100g,"874,557",0,84,0.00,0.01
sugars_100g,"874,227",0,53,0.00,0.01
fiber_100g,"375,761",62,18,0.02,0.00
proteins_100g,"875,886",1,33,0.00,0.00
salt_100g,"821,330",0,60,0.00,0.01
sodium_100g,"821,330",0,32,0.00,0.00
fruits-vegetables-legumes_100g,"347,951",131,"1,023",0.04,0.29


In [22]:
tol = 0.5  # marge pour les arrondis d'étiquettes

sug = dataframe[["sugars_100g", "carbohydrates_100g"]].dropna()
sat = dataframe[["saturated-fat_100g", "fat_100g"]].dropna()
mac = dataframe[["fat_100g", "carbohydrates_100g", "proteins_100g", "salt_100g"]].dropna()

rules = {
    "sucres > glucides": ((sug["sugars_100g"] > sug["carbohydrates_100g"] + tol).sum(), len(sug)),
    "saturés > graisses": ((sat["saturated-fat_100g"] > sat["fat_100g"] + tol).sum(), len(sat)),
    "graisses+glucides+protéines+sel > 105": ((mac.sum(axis=1) > 105).sum(), len(mac)),
}

table = pd.DataFrame(rules, index=["violations", "comparables"]).T
table["%"] = table["violations"] / table["comparables"] * 100
table.style.format({"violations": "{:,.0f}", "comparables": "{:,.0f}", "%": "{:.2f}"})

,violations,comparables,%
sucres > glucides,450,"868,048",0.05
saturés > graisses,220,"865,680",0.03
graisses+glucides+protéines+sel > 105,"1,220","809,391",0.15


In [23]:
cat = dataframe.select_dtypes(include=["object", "string"])

card = pd.DataFrame({
    "distinctes": cat.nunique(),
    "% manquants": cat.isna().mean() * 100,
})
card.index.name = "colonne"

card.sort_values("distinctes").style.format({"distinctes": "{:,.0f}", "% manquants": "{:.1f}"})

,distinctes,% manquants
colonne,,
nutriscore_grade,7,2.1
product_name,"732,991",10.4
code,"1,261,237",0.0


In [26]:
products = pd.read_csv(
    CSV_GZ,
    sep="\t",
    usecols=["code", "pnns_groups_1", "pnns_groups_2", "nutriscore_grade"],
    dtype={"code": "string", "pnns_groups_1": "category", "pnns_groups_2": "category"},
    low_memory=False,
)

products["pnns_groups_1"].value_counts(dropna=False)

pnns_groups_1
unknown                    673295
Sugary snacks              117808
Fish Meat Eggs             100566
Milk and dairy products     64340
Cereals and potatoes        57320
Beverages                   46428
Composite foods             42081
Fruits and vegetables       41400
Fat and sauces              40066
Salty snacks                31649
NaN                         25924
Alcoholic beverages         17833
Baby foods                   2553
Name: count, dtype: int64

In [27]:
products["scored"] = products["nutriscore_grade"].isin(list("abcde"))

by_group = (
    products
    .groupby("pnns_groups_1", observed=True, dropna=False)
    .agg(produits=("code", "size"), part_notes=("scored", "mean"))
    .sort_values("produits", ascending=False)
)

by_group.style.format({"produits": "{:,.0f}", "part_notes": "{:.1%}"})

,produits,part_notes
pnns_groups_1,,
unknown,"673,295",5.7%
Sugary snacks,"117,808",79.3%
Fish Meat Eggs,"100,566",79.5%
Milk and dairy products,"64,340",84.8%
Cereals and potatoes,"57,320",82.6%
Beverages,"46,428",71.9%
Composite foods,"42,081",89.9%
Fruits and vegetables,"41,400",73.2%
Fat and sauces,"40,066",79.0%


In [ ]:
NUTRIENTS = [
    "energy-kcal_100g", "fat_100g", "saturated-fat_100g", "sugars_100g",
    "fiber_100g", "proteins_100g", "salt_100g",
]

completeness = dataframe[NUTRIENTS].notna().sum(axis=1)
completeness.value_counts().sort_index()

0    366505
1     12189
2      5654
3      3095
4      3110
5     41131
6    482145
7    347434
Name: count, dtype: int64

In [29]:
completeness_counts = completeness.value_counts().sort_index()

completeness_table = pd.DataFrame({
    "produits": completeness_counts,
    "%": completeness_counts / completeness_counts.sum() * 100,
})
completeness_table["produits avec au moins N"] = completeness_table["produits"][::-1].cumsum()[::-1]
completeness_table["% avec au moins N"] = (
    completeness_table["produits avec au moins N"] / len(completeness) * 100
)
completeness_table.index.name = f"nutriments renseignés (sur {len(NUTRIENTS)})"

completeness_table.style.format({
    "produits": "{:,.0f}",
    "%": "{:.1f}",
    "produits avec au moins N": "{:,.0f}",
    "% avec au moins N": "{:.1f}",
})

,produits,%,produits avec au moins N,% avec au moins N
nutriments renseignés (sur 7),,,,
0,"366,505",29.1,"1,261,263",100.0
1,"12,189",1.0,"894,758",70.9
2,"5,654",0.4,"882,569",70.0
3,"3,095",0.2,"876,915",69.5
4,"3,110",0.2,"873,820",69.3
5,"41,131",3.3,"870,710",69.0
6,"482,145",38.2,"829,579",65.8
7,"347,434",27.5,"347,434",27.5
